In [ ]:
"""
exp196: H-070追加検証#6 — RealMLP、高欠損率3列(sleep_duration/calorie_expenditure/water_intake)の
was_missingフラグを除去し、TEのnaカテゴリで欠損情報を代替できるか検証 — Kaggle Notebook 自己完結版

FULL(exp187: 13base+7フラグ+39TE)と比較。フラグ3列除去以外はexp187と同一構成。
train_features_slim.pkl / test_features_slim.pkl は Dataset kakiginobuya/s6e7-features-slim
（2026-07-14更新版、39個のte_exact_*列を含む）から読み込む。

出力: /kaggle/working/oof_196_realmlp_noflag.npy, /kaggle/working/test_196_realmlp_noflag.npy
"""

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

import subprocess
import sys


def _gpu_name():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                              capture_output=True, text=True, timeout=10)
        return out.stdout.strip()
    except Exception:
        return ""

gpu_name = _gpu_name()
print(f"detected GPU: {gpu_name!r}")
if "P100" in gpu_name:
    print("P100 detected -> reinstalling torch cu118 build (retains Pascal/sm_60 support)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "torch==2.3.1", "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], check=True)

try:
    import pytabkit  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pytabkit"], check=True)
    for dep in ("pytorch-lightning", "torchmetrics"):
        try:
            __import__(dep.replace("-", "_"))
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", dep], check=True)

from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier

DATASET_NAME = "s6e7-features-slim"
_KAGGLE_INPUT = Path("/kaggle/input")
_ds_candidates = [
    _KAGGLE_INPUT / "datasets" / "kakiginobuya" / DATASET_NAME,
    _KAGGLE_INPUT / DATASET_NAME,
]
DATASET_DIR = next((p for p in _ds_candidates if p.exists()), _ds_candidates[0])
print(f"DATASET_DIR = {DATASET_DIR}")

TARGET_COL = "health_condition"
N_CLASSES = 3
N_SPLITS = 5
SEED = 42

BASE_FEATURES = [
    "sleep_duration_raw_nan", "heart_rate_raw_nan", "bmi_raw_nan", "calorie_expenditure_raw_nan",
    "step_count_raw_nan", "exercise_duration_raw_nan", "water_intake_raw_nan",
    "gender", "physical_activity_level", "sleep_quality", "smoking_alcohol", "stress_level", "diet_type",
]
# 欠損率上位3の数値列: sleep_duration(11.01%)/calorie_expenditure(7.66%)/water_intake(6.30%)の
# was_missingフラグを除去(TEのnaカテゴリに欠損情報を委ねる仮説の検証)
NO_FLAG_COLS = {"sleep_duration_raw_nan", "calorie_expenditure_raw_nan", "water_intake_raw_nan"}


def main():
    train = pd.read_pickle(DATASET_DIR / "train_features_slim.pkl")
    test = pd.read_pickle(DATASET_DIR / "test_features_slim.pkl")
    print(f"train: {train.shape}, test: {test.shape}")

    te_cols = [c for c in train.columns if c.startswith("te_exact_")]
    FEATURES = BASE_FEATURES + te_cols
    print(f"{len(FEATURES)} features ({len(BASE_FEATURES)} base + {len(te_cols)} TE)")

    X, y_raw = train[FEATURES].copy(), train[TARGET_COL]
    X_test = test[FEATURES].copy()

    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y_raw), index=y_raw.index)
    classes = le.classes_

    cat_cols = [c for c in X.columns if str(X[c].dtype) in ("category", "object")]
    for c in cat_cols:
        X[c] = X[c].astype(str)
        X_test[c] = X_test[c].astype(str)

    num_cols = [c for c in X.columns if c not in cat_cols]
    skipped_flags = []
    for c in num_cols:
        if X[c].isna().any() or X_test[c].isna().any():
            med = X[c].median()
            if c in NO_FLAG_COLS:
                skipped_flags.append(c)
                X[c] = X[c].fillna(med)
                X_test[c] = X_test[c].fillna(med)
            else:
                flag_col = f"{c}_was_missing"
                X[flag_col] = X[c].isna().astype(str)
                X_test[flag_col] = X_test[c].isna().astype(str)
                cat_cols.append(flag_col)
                X[c] = X[c].fillna(med)
                X_test[c] = X_test[c].fillna(med)

    print("cat_cols:", cat_cols)
    print("skipped was_missing flags for:", skipped_flags)
    print("num_cols:", num_cols)

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_preds = np.zeros((len(train), N_CLASSES))
    test_preds = np.zeros((len(test), N_CLASSES))
    train_scores, val_scores = [], []

    t0 = time.time()
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = RealMLP_TD_Classifier(device="cuda", random_state=SEED, verbosity=1)
        model.fit(X_tr, y_tr, X_val, y_val, cat_col_names=cat_cols)

        val_pred = model.predict_proba(X_val)
        oof_preds[val_idx] = val_pred
        test_preds += model.predict_proba(X_test) / N_SPLITS

        tr_pred = model.predict_proba(X_tr)
        tr_score = balanced_accuracy_score(y_tr, np.argmax(tr_pred, axis=1))
        val_score = balanced_accuracy_score(y_val, np.argmax(val_pred, axis=1))
        train_scores.append(tr_score)
        val_scores.append(val_score)
        print(f"fold {fold}: train={tr_score:.5f} val={val_score:.5f} elapsed={time.time()-t0:.1f}s")

    oof_score = balanced_accuracy_score(y, np.argmax(oof_preds, axis=1))
    print(f"\nRealMLP +H-070 noflag(3cols) raw argmax OOF = {oof_score:.5f}")
    print(f"train_mean={np.mean(train_scores):.5f}±{np.std(train_scores):.5f} "
          f"val_mean={np.mean(val_scores):.5f}±{np.std(val_scores):.5f} "
          f"gap={np.mean(train_scores)-np.mean(val_scores):.5f}")
    print(f"total elapsed = {(time.time()-t0)/60:.1f} min")

    prior = pd.Series(y_raw).value_counts().reindex(classes).to_numpy() / len(y_raw)
    BETA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.15, 1.3, 1.5, 1.75, 2.0, 2.5]
    scores = {}
    for b in BETA_GRID:
        pred = classes[(oof_preds / prior**b).argmax(1)]
        scores[b] = balanced_accuracy_score(y_raw, pred)
        print(f"beta={b:>4}: calibrated OOF = {scores[b]:.5f}")
    best_b = max(scores, key=scores.get)
    print(f"\nbest beta={best_b}, calibrated OOF={scores[best_b]:.5f}")
    print(f"FULL(with all flags, exp187) = 0.94935")
    print(f"Delta vs FULL = {scores[best_b] - 0.94935:+.5f}")

    np.save("/kaggle/working/oof_196_realmlp_noflag.npy", oof_preds)
    np.save("/kaggle/working/test_196_realmlp_noflag.npy", test_preds)
    print("saved: /kaggle/working/oof_196_realmlp_noflag.npy, /kaggle/working/test_196_realmlp_noflag.npy")


if __name__ == "__main__":
    main()
